# 04 — Warm-Start Fine-Tune
Fine-tunes Qwen2.5-Coder on all knowledge pairs extracted by notebooks 03 and 05
to give the model a solid understanding of ARO syntax before the RL explore loop.

Uses LoRA with conservative settings (8 layers, LR 1e-5, max-seq-length 2048,
grad-checkpoint) to avoid gradient instability on long sequences.
Training progress is shown as a live loss graph and progress bar.

**Run after:** notebook 03 (and optionally notebook 05 for book Q&A pairs)

**Input:**  `../data/02_knowledge/knowledge_pairs.jsonl`  (from notebooks 03 + 05)
            `../data/02_knowledge/knowledge.json`          (for system prompt)
**Output:** `../data/adapters/warm_start/`                 (LoRA adapter)
            `../data/02_knowledge/knowledge.json`          (updated with adapter path)

In [5]:
import gc, json, re, random, subprocess, sys
from pathlib import Path
from collections import Counter

def ensure_mlx_lm():
    try:
        from mlx_lm import load, generate as mlx_generate
        from mlx_lm.sample_utils import make_sampler
        return load, mlx_generate, make_sampler
    except ModuleNotFoundError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlx-lm'], check=True)
        from mlx_lm import load, generate as mlx_generate
        from mlx_lm.sample_utils import make_sampler
        return load, mlx_generate, make_sampler

load, mlx_generate, make_sampler = ensure_mlx_lm()

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

DATA_DIR    = SCRIPT_DIR / '../data/02_knowledge'
PAIRS_FILE  = DATA_DIR / 'knowledge_pairs.jsonl'
ADAPTER_DIR = SCRIPT_DIR / '../data/adapters'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'mlx-community/Qwen2.5-Coder-7B-Instruct-4bit'

with open(DATA_DIR / 'knowledge.json') as f:
    kb = json.load(f)

print(f'Knowledge pairs: {PAIRS_FILE}')
print(f'Adapter output:  {ADAPTER_DIR}')

Knowledge pairs: /Users/kris/Projects/ARO-App/Train/script/../data/02_knowledge/knowledge_pairs.jsonl
Adapter output:  /Users/kris/Projects/ARO-App/Train/script/../data/adapters


In [6]:
# Load all knowledge pairs produced by notebooks 03 and 05
all_pairs = []
with open(PAIRS_FILE) as f:
    for line in f:
        if line.strip():
            try:
                all_pairs.append(json.loads(line))
            except Exception:
                pass

sources = Counter(p['source'].split(':')[0] for p in all_pairs)
scores  = Counter(round(p.get('score', 1.0), 1) for p in all_pairs)

print(f'Total pairs: {len(all_pairs)}')
print('\nBy source:')
for src, n in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'  {src:30s}: {n}')
print('\nBy score:')
for score, n in sorted(scores.items(), key=lambda x: -x[0]):
    print(f'  {score}: {n}')

Total pairs: 396

By source:
  example                       : 237
  proposal                      : 115
  aro_by_example                : 23
  book                          : 21

By score:
  1.0: 396


In [7]:
# Build system prompt from action metadata (same prompt used in notebooks 03 and 06)
action_lines = []
for a in kb['actions']:
    if a['verbs']:
        v = '/'.join(a['verbs'][:3])
        p = ', '.join(a['prepositions'][:4])
        action_lines.append(f'- {v}  (role: {a["role"]}, prepositions: {p})')

SYSTEM_PROMPT = f"""You are an expert ARO language programmer.
ARO (Action Result Object) is a DSL for business logic as natural-language statements.

SYNTAX:
  (FeatureSetName: BusinessActivity) {{
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }}

KEY RULES:
- Articles (a/an/the) are optional
- String concatenation: ++ (NOT +)
- Variable names: hyphenated lowercase e.g. <user-id>, <order-total>
- Forbidden variable name prefixes: is-, with-, empty-
- Exactly ONE Application-Start per application
- openapi.yaml required for HTTP server; operationId must match feature set name
- Long-running apps: Keepalive the <application> for the <events>.
- Return statuses: <OK: status>, <Created: status>, <NotFound: status>
- HTTP path params: Extract the <id> from the <pathParameters: id>.
- Request body:     Extract the <data> from the <request: body>.

AVAILABLE ACTIONS:
{chr(10).join(action_lines[:40])}

Always wrap ARO code in ```aro ... ``` fences."""

print(f'System prompt: {len(SYSTEM_PROMPT)} chars')

System prompt: 1939 chars


## Warm-Start Fine-Tune

Fine-tune Qwen2.5-Coder on all extracted pairs so it understands ARO syntax
before notebook 06 starts generating synthetic data.

Uses 8 LoRA layers (same as the RL loop in notebook 06) with a conservative
learning rate of 1e-5 to prevent gradient instability on long sequences.
Sequences are truncated to 2048 tokens and `--grad-checkpoint` reduces peak memory.

The adapter is saved to `../data/adapters/warm_start/` and notebook 06
will automatically load it as the starting point for RL fine-tuning rounds.

In [ ]:
import matplotlib.pyplot as plt
from IPython import display as ipydisplay
try:
    import ipywidgets                    # raises ImportError if not installed
    from tqdm.notebook import tqdm       # rich widget bar
except ImportError:
    from tqdm import tqdm                # plain text fallback

WARM_ADAPTER = ADAPTER_DIR / 'warm_start'
SFT_DIR      = SCRIPT_DIR / '../data/warm_start_sft'
SFT_DIR.mkdir(parents=True, exist_ok=True)

# Shuffle and split
random.shuffle(all_pairs)
split = max(1, int(len(all_pairs) * 0.1))

def pair_to_chat(p):
    return {'messages': [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': p['instruction']},
        {'role': 'assistant', 'content': p['output']},
    ]}

(SFT_DIR / 'valid.jsonl').write_text(
    '\n'.join(json.dumps(pair_to_chat(p)) for p in all_pairs[:split]))
(SFT_DIR / 'train.jsonl').write_text(
    '\n'.join(json.dumps(pair_to_chat(p)) for p in all_pairs[split:]))

n_train = len(all_pairs) - split
# ~0.04 it/sec on M-series; cap at 300 iters (~2 hours) to stay practical.
# 3 epochs over training set, never more than 300.
iters   = max(100, min(300, n_train * 3))

print(f'SFT data: {n_train} train, {split} valid')
print(f'Warm-start: {iters} steps, 8 LoRA layers')

cmd = [
    sys.executable, '-m', 'mlx_lm', 'lora',
    '--model',          MODEL_ID,
    '--data',           str(SFT_DIR),
    '--train',
    '--num-layers',     '8',          # fewer layers → more stable gradients
    '--iters',          str(iters),
    '--batch-size',     '2',
    '--learning-rate',  '1e-5',       # conservative LR to avoid NaN loss
    '--adapter-path',   str(WARM_ADAPTER),
    '--mask-prompt',
    '--max-seq-length', '2048',       # truncate long sequences before they destabilise gradients
    '--grad-checkpoint',              # reduce peak memory usage
    '--save-every',     str(max(50, iters // 5)),
    '--val-batches',    '5',
]

# ── Live loss graph ──────────────────────────────────────────────────────────
train_iters, train_losses = [], []
val_iters,   val_losses   = [], []

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('Warm-start Fine-tune — Loss')
ax.grid(True, alpha=0.3)
train_line, = ax.plot([], [], 'b-o', ms=3, lw=1.5, label='Train')
val_line,   = ax.plot([], [], 'r-s', ms=6, lw=1.5, label='Val')
ax.legend()
plt.tight_layout()

_fig_handle = ipydisplay.display(fig, display_id=True)

def _refresh_plot():
    if train_losses:
        train_line.set_data(train_iters, train_losses)
    if val_losses:
        val_line.set_data(val_iters, val_losses)
    all_y = train_losses + val_losses
    if all_y:
        lo, hi = min(all_y), max(all_y)
        pad = max(0.05, (hi - lo) * 0.15)
        ax.set_ylim(lo - pad, hi + pad)
    all_x = train_iters + val_iters
    if all_x:
        ax.set_xlim(0, max(iters, max(all_x)) + 5)
    fig.canvas.draw()
    _fig_handle.update(fig)

# ── Regexes for mlx-lm log lines ────────────────────────────────────────────
# Train: "Iter 10: Train loss 1.234, Learning Rate 1.00e-05, It/sec 0.042,
#          Tokens/sec 85.3, Trained Tokens 20480, Peak mem 22.5 GB"
# Val:   "Iter 10: Val loss 0.796, Val took 62.382s"
_train_re  = re.compile(
    r'Iter\s+(\d+):\s+Train loss\s+([\d.]+)'
    r'(?:.*?Learning Rate\s+([\d.e+-]+))?'
    r'(?:.*?It/sec\s+([\d.]+))?'
    r'(?:.*?Tokens/sec\s+([\d.]+))?'
    r'(?:.*?Trained Tokens\s+([\d,]+))?'
    r'(?:.*?Peak mem\s+([\d.]+)\s*GB)?'
)
_val_re    = re.compile(r'Iter\s+(\d+):\s+Val loss\s+([\d.]+)(?:.*?Val took\s+([\d.]+)s)?')
_saved_re  = re.compile(r'Adapter saved', re.IGNORECASE)

# ── Progress bar + subprocess ────────────────────────────────────────────────
pbar      = tqdm(total=iters, desc='Fine-tuning', unit='iter', dynamic_ncols=True)
last_iter = 0

print('Running:', ' '.join(cmd))
proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for raw_line in proc.stdout:
        line = raw_line.rstrip()

        m_train = _train_re.search(line)
        m_val   = _val_re.search(line)

        if m_train:
            it          = int(m_train.group(1))
            loss        = float(m_train.group(2))
            lr          = m_train.group(3)
            it_sec      = m_train.group(4)
            tok_sec     = m_train.group(5)
            trained_tok = m_train.group(6)
            peak_mem    = m_train.group(7)

            train_iters.append(it)
            train_losses.append(loss)
            pbar.update(it - last_iter)
            last_iter = it

            # Compute ETA
            eta_str = ''
            if it_sec:
                remaining = iters - it
                eta_s = remaining / float(it_sec)
                h, m_ = divmod(int(eta_s), 3600)
                m_, s  = divmod(m_, 60)
                eta_str = f'{h}h{m_:02d}m' if h else f'{m_}m{s:02d}s'

            # Update progress bar postfix
            postfix = {'loss': f'{loss:.3f}'}
            if it_sec:  postfix['it/s']    = it_sec
            if peak_mem: postfix['mem_GB'] = peak_mem
            if eta_str:  postfix['ETA']    = eta_str
            pbar.set_postfix(postfix)

            # Print a formatted info line
            parts = [f'iter {it:>4}/{iters}', f'train_loss {loss:.4f}']
            if lr:          parts.append(f'lr {float(lr):.2e}')
            if it_sec:      parts.append(f'{float(it_sec):.3f} it/s')
            if tok_sec:     parts.append(f'{float(tok_sec):.0f} tok/s')
            if trained_tok: parts.append(f'{trained_tok.replace(",","")} tokens')
            if peak_mem:    parts.append(f'mem {peak_mem} GB')
            if eta_str:     parts.append(f'ETA {eta_str}')
            tqdm.write('  ' + '  │  '.join(parts))

            _refresh_plot()

        elif m_val:
            it       = int(m_val.group(1))
            loss     = float(m_val.group(2))
            val_took = m_val.group(3)

            val_iters.append(it)
            val_losses.append(loss)

            postfix = {'loss': f'{train_losses[-1]:.3f}' if train_losses else '?',
                       'val':  f'{loss:.3f}'}
            pbar.set_postfix(postfix)

            took_str = f'  ({val_took}s)' if val_took else ''
            tqdm.write(f'  ── val ──  iter {it:>4}/{iters}  val_loss {loss:.4f}{took_str}')

            _refresh_plot()

        elif _saved_re.search(line):
            tqdm.write(f'  ✓ {line}')

        else:
            # Echo warnings, loading messages, etc.
            if line.strip():
                tqdm.write(f'  {line}')

finally:
    proc.wait()
    pbar.close()

_refresh_plot()   # final draw

if proc.returncode == 0:
    print(f'\nWarm-start adapter saved to: {WARM_ADAPTER}')
else:
    print(f'\nFine-tune exited with code {proc.returncode}')


In [ ]:
# Update knowledge.json so notebook 06 finds the warm-start adapter automatically
kb['warm_start_adapter']    = str(WARM_ADAPTER)
kb['knowledge_pairs_file']  = str(PAIRS_FILE)
kb['knowledge_pairs_count'] = len(all_pairs)

with open(DATA_DIR / 'knowledge.json', 'w') as f:
    json.dump(kb, f, indent=2)

print('Updated knowledge.json')
print()
print('Next steps:')
print(f'  Adapter path: {WARM_ADAPTER}')
print(f'  Run notebook 06 — it will auto-load this adapter and run the RL explore loop')